In [ ]:
import altair as alt
import polars as pl

In [ ]:
#file = "/master/abagwell/variant-analysis/results/rhesus/gwas/U42_WGS_WES.fastGWA_GLMM.fastGWA"
#file = "/master/abagwell/variant-analysis/results/rhesus/gwas/pruned/U42_WGS_WES.RPL_non-RPL_females_WGS.fastGWA_GLMM.fastGWA"
file = "/master/abagwell/variant-analysis/results/rhesus/gwas/impactful/U42_WGS_WES.RPL_non-RPL_females_WGS.fastGWA_GLMM.fastGWA"

min_y = 0

table = pl.read_csv(file, separator="\t", infer_schema_length=10000).with_columns(
    locus = pl.concat_str([
        pl.col("CHR"),
        pl.lit(":"),
        pl.col("POS"),
        ]),
    pval_log10 = pl.col("P").log10().abs()
).filter(
    (pl.col("P") != 0.0) &
    (pl.col('pval_log10') > min_y)
)
# .with_columns(
#     pl.col("Pvalue").log10().alias("pval_log10").abs()  #.mul(-1)
# )

In [ ]:
table

In [ ]:
# df = pl.read_csv("/master/abagwell/variant-analysis/results/rhesus/genotypes/impactful/U42_WGS_WES.RPL_non-RPL_females_WGS_0.01.SNP.chr10.bcf",
#             has_header=False,
#             separator="\t",
#             comment_prefix="#",
#             truncate_ragged_lines=True)
# df
# import sgkit as sg

# bcfs = ["/master/abagwell/variant-analysis/results/rhesus/genotypes/impactful/U42_WGS_WES.RPL_non-RPL_females_WGS_0.01.SNP.chr{chrom}.bcf" for chrom in range(1,21)]
# zarr = "/master/abagwell/variant-analysis/results/rhesus/genotypes/impactful/U42_WGS_WES.RPL_non-RPL_females_WGS_0.01.SNP.zarr"

# sg.io.vcf.vcf_to_zarr(bcfs, zarr)
# ds = sg.load_dataset(zarr)

import allel
import polars as pl



df = allel.vcf_to_dataframe("/master/abagwell/variant-analysis/results/rhesus/genotypes/impactful/U42_WGS_WES.RPL_non-RPL_females_WGS_0.01.SNP.autosomal.vcf.gz", fields=['variants/CHROM', 'variants/POS', 'variants/IMPACT'])





In [ ]:
pl.from_pandas(df)

In [ ]:
# Condensed

# alt.data_transformers.disable_max_rows()
# plot = alt.Chart(table).mark_circle().encode(
#     alt.X("locus:O", title="Locus", axis=alt.Axis(labels=False, tickSize=0), sort=['1','2','3','4','5']),
#     alt.Y("pval_log10:Q", title="-log10(p)", scale=alt.Scale(domain=[2.5, round(table['pval_log10'].max() + 0.5)])),
#     color=alt.Color("CHR:N", title="CHROM", sort=['1','2','3','4','5']),
#     tooltip=[
#         alt.Tooltip("locus:O", title="Locus"),
#         alt.Tooltip("pval_log10:Q", title="-log10(p)"),
#     ],
# ).properties(
#     title="fastGWA_GLMM in WGS RPL vs non-RPL",
#     width=500,
# )

In [ ]:
chromosomes = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20"]
chromosome_lengths = [223_616_942, 196_197_964, 185_288_947, 169_963_040, 187_317_192, 179_085_566, 169_868_564, 145_679_320, 134_124_166, 99_517_758, 133_066_086, 130_043_856, 108_737_130, 128_056_306, 113_283_604, 79_627_064, 95_433_459, 74_474_043, 58_315_233, 77_137_495]  # For Mmul_10

alt.data_transformers.disable_max_rows()

plots_by_chrom = []
for chrom in chromosomes:
    if chrom == "1":
        axis = alt.Axis(tickSize=0)
    else:
        axis = alt.Axis(labels=False, tickSize=0, title="", domainOpacity=0)

    plot = alt.Chart(table.filter(pl.col("CHR") == int(chrom))).mark_circle().encode(
        alt.X("POS:O", title="Position", axis=alt.Axis(labels=False, tickSize=0), sort=['1','2','3','4','5']),
        alt.Y("pval_log10:Q", title="-log10(p)", scale=alt.Scale(domain=[min_y, round(table['pval_log10'].max() + 0.5)]), axis=axis),
        color=alt.Color("CHR:N", title="CHROM", sort=['1','2','3','4','5'], legend=None),
        tooltip=[
            alt.Tooltip("locus:O", title="Locus"),
            alt.Tooltip("pval_log10:Q", title="-log10(p)"),
        ],
    ).properties(
            width=chromosome_lengths[int(chrom) - 1]/1500000,
            height=alt.Step(10),
            #width=300, # For when only displaying one chromosome
            title=[f"chr{chrom}"]
    )
    plots_by_chrom.append(plot)

    


In [ ]:
concat = alt.hconcat(*plots_by_chrom, spacing=0).properties(title="fastGWA-GLMM in WGS RPL vs non-RPL")
concat.save("/master/abagwell/figures/gwas/fastGWA_GLMM.html")
concat.save("/master/abagwell/figures/gwas/fastGWA_GLMM.svg")

In [ ]:
concat

In [ ]:
#concat.save("/master/abagwell/figures/gwas/fastGWA_GLMM.pruned.svg")

In [ ]:
# # For fastGWA-BB
# file = "/master/abagwell/variant-analysis/results/rhesus/gwas/U42_WGS_WES.fastGWA_BB.fastGWA"

# table = pl.read_csv(file, separator="\t", infer_schema_length=10000).with_columns(
#     interval = pl.concat_str([
#         pl.col("START"),
#         pl.lit(":"),
#         pl.col("END"),
#         ]),
#     pval_log10 = pl.col("P").log10().abs()
# ).filter(
#     (pl.col("P") != 0.0) &
#     (pl.col('pval_log10') > 2.5)
# )
# # .with_columns(
# #     pl.col("Pvalue").log10().alias("pval_log10").abs()  #.mul(-1)
# # )

In [ ]:
# gtf
import altair as alt
from gtfparse import read_gtf
import polars as pl

gtf = "/master/abagwell/variant-analysis/resources/rhesus_prev/annotations/Macaca_mulatta.Mmul_10.110.gtf.gz"
df = read_gtf(gtf)

gene_names = df.filter(
    pl.col("feature") == "gene"
).select(
    "gene_id", "gene_name"
)

#genes_of_interest = pl.read_csv("/master/abagwell/variant-analysis/results/rhesus/gwas/genes/RPL_gene_names.list", has_header=False, separator="\t", new_columns=["gene"])
genes_of_interest = pl.read_csv("/master/abagwell/variant-analysis/results/rhesus/gwas/genes/RPL_ensembl_ids.list", has_header=False, separator="\t", new_columns=["gene"])

In [ ]:
chromosomes = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20"]
chromosome_lengths = [223_616_942, 196_197_964, 185_288_947, 169_963_040, 187_317_192, 179_085_566, 169_868_564, 145_679_320, 134_124_166, 99_517_758, 133_066_086, 130_043_856, 108_737_130, 128_056_306, 113_283_604, 79_627_064, 95_433_459, 74_474_043, 58_315_233, 77_137_495]  # For Mmul_10


tables = []
for chrom in chromosomes:

    #file = f"/master/abagwell/variant-analysis/results/rhesus/gwas/pass/U42_WGS_WES.RPL_non-RPL_females_WGS.fastGWA_BB.chr{chrom}.fastGWA"
    file = f"/master/abagwell/variant-analysis/results/rhesus/gwas/impactful/U42_WGS_WES.RPL_non-RPL_females_WGS_0.01.fastGWA_BB.chr{chrom}.fastGWA"
    table = pl.read_csv(file, separator="\t", infer_schema_length=10000).with_columns(
        interval = pl.concat_str([
            pl.col("START"),
            pl.lit(":"),
            pl.col("END"),
            ]),
        pval_log10 = pl.col("P").log10().abs(),
        chr=pl.lit(chrom)
    ).join(gene_names, how="left", left_on="GENE", right_on="gene_id"
    #).join(genes_of_interest, how="inner", left_on="gene_name", right_on="gene")
    ).join(genes_of_interest, how="inner", left_on="GENE", right_on="gene")
    tables.append(table)
max = pl.concat(tables)["pval_log10"].max()

In [ ]:
pl.concat(tables).write_csv("/master/abagwell/figures/gwas/gwas.BB.tsv", separator="\t")


In [ ]:
pl.concat(tables)

In [ ]:
chromosomes = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20"]
chromosome_lengths = [223_616_942, 196_197_964, 185_288_947, 169_963_040, 187_317_192, 179_085_566, 169_868_564, 145_679_320, 134_124_166, 99_517_758, 133_066_086, 130_043_856, 108_737_130, 128_056_306, 113_283_604, 79_627_064, 95_433_459, 74_474_043, 58_315_233, 77_137_495]  # For Mmul_10

alt.data_transformers.disable_max_rows()


colors = [
  '#5778a4',
  '#e49444',
  '#d1615d',
  '#85b6b2',
  '#6a9f58',
  '#e7ca60',
  '#a87c9f',
  '#f1a2a9',
  '#967662',
  '#b8b0ac',
  # Next cycle
  '#5778a4',
  '#e49444',
  '#d1615d',
  '#85b6b2',
  '#6a9f58',
  '#e7ca60',
  '#a87c9f',
  '#f1a2a9',
  '#967662',
  '#b8b0ac',
]



plots_by_chrom = []
for table, color in zip(tables, colors):
    chrom = table["chr"][0]
    #file = f"/master/abagwell/variant-analysis/results/rhesus/gwas/U42_WGS_WES.fastGWA_BB.chr{chrom}.fastGWA"
    #file = f"/master/abagwell/variant-analysis/results/rhesus/gwas/pass/U42_WGS_WES.RPL_non-RPL_females_WGS.fastGWA_BB.chr{chrom}.fastGWA"
    #file = f"/master/abagwell/variant-analysis/results/rhesus/gwas/impactful/U42_WGS_WES.RPL_non-RPL_females_WGS_0.01.fastGWA_BB.chr{chrom}.fastGWA"

    # .filter(
    #     (pl.col("P") != 0.0) &
    #     (pl.col('pval_log10') > 2.5)
    # )

    if chrom == "1":
        axis = alt.Axis(tickSize=0)
    else:
        axis = alt.Axis(labels=False, tickSize=0, title="", domainOpacity=0)

    plot = alt.Chart(table).mark_circle(color=color).encode(
        alt.X("START:O", title="Position", axis=alt.Axis(labels=False, tickSize=0), sort=['1','2','3','4','5']),
        alt.Y("pval_log10:Q", title="-log10(p)", scale=alt.Scale(domain=[0, max]), axis=axis),
        #alt.Y("P:Q", title="-log10(p)", scale=alt.Scale(domain=[0, max]), axis=axis),
        #color=alt.Color("CHR:N", title="CHROM", sort=['1','2','3','4','5'], legend=None),
        #color="#f58518",
        tooltip=[
            alt.Tooltip("GENE", title="ENSEMBL ID"),
            alt.Tooltip("gene_name", title="Gene"),
            alt.Tooltip("VAR_N:Q", title="# of Variants"),
            alt.Tooltip("interval:O", title="Interval"),
            alt.Tooltip("pval_log10:Q", title="-log10(p)"),
            alt.Tooltip("P:Q", title="p-value"),
        ],
    ).properties(
            width=chromosome_lengths[int(chrom) - 1]/1500000,
            height=alt.Step(10),
            #width=300, # For when only displaying one chromosome
            title=[f"chr{chrom}"]
    )
    plots_by_chrom.append(plot)


In [ ]:
plot = alt.hconcat(*plots_by_chrom, spacing=0).properties(title="fastGWA-BB in WGS RPL vs non-RPL")

In [ ]:
plot

In [ ]:
plot.save("/master/abagwell/figures/gwas/fastGWA_BB.goi.html")
plot.save("/master/abagwell/figures/gwas/fastGWA_BB.goi.svg")

In [ ]:
tables[-1]